# 练习 4：Pydantic 数据校验

**AI 应用场景**

- 定义 LLM 工具（function calling）的参数结构
- 结构化输出：要求模型必须返回指定格式的 JSON
- OpenAI 官方 SDK 内部就是用 Pydantic 做校验的

**核心语法**

| 写法 | 作用 |
|------|------|
| `class Xxx(BaseModel)` | 定义数据模型 |
| `Field(description=...)` | 字段约束 + 给 LLM 看的描述 |
| `model_dump()` | 转成普通 dict |
| 类型不对 | 自动报错，不用手写 if 检查 |

In [1]:
from pydantic import BaseModel, Field, ValidationError

class WeatherTool(BaseModel):
    # 天气查询工具的参数定义（模拟 function calling 的 JSON Schema）
    city: str = Field(description="城市名称，例如：北京")
    days: int = Field(default=1, ge=1, le=7, description="查询天数，1~7")
    unit: str = Field(default="celsius", description="温度单位")

print("WeatherTool 已定义 ✅")

WeatherTool 已定义 ✅


In [2]:
# 正确用法：自动类型转换 + 默认值
tool = WeatherTool(city="上海", days=3)
print(f"模型对象: {tool}")
print(f"转成 dict: {tool.model_dump()}")

模型对象: city='上海' days=3 unit='celsius'
转成 dict: {'city': '上海', 'days': 3, 'unit': 'celsius'}


In [3]:
# 错误用法：days=99 超出 le=7 的限制，Pydantic 自动拦截
try:
    WeatherTool(city="北京", days=99)
except ValidationError as e:
    print(f"✅ Pydantic 拦截: {e.errors()[0]['msg']}")

✅ Pydantic 拦截: Input should be less than or equal to 7


## 结构化输出：让 LLM 必须按这个格式回答

In [4]:
class SearchResult(BaseModel):
    # 结构化输出的模型定义
    answer: str = Field(description="回答内容")
    confidence: float = Field(ge=0.0, le=1.0, description="置信度 0~1")
    sources: list[str] = Field(description="引用来源列表")

result = SearchResult(
    answer="报销需要先在 OA 系统提交申请，审批通过后财务 3 天内打款。",
    confidence=0.9,
    sources=["employee_handbook.txt"],
)
print(result.model_dump())

{'answer': '报销需要先在 OA 系统提交申请，审批通过后财务 3 天内打款。', 'confidence': 0.9, 'sources': ['employee_handbook.txt']}


## 💡 小结

- `Field(description=...)` 会直接变成给 LLM 看的 JSON Schema —— 你项目 `03_function_calling/tools.py` 里手写的 JSON Schema，用 Pydantic 可以自动生成
- 面试常问：**"怎么保证 LLM 输出的 JSON 是合法的？"** → Pydantic 校验 / 失败重试 / 结构化输出 API